# Amazon ML Challenge 2026: Business Entity Resolution (v2)

For every Source-1 (S1) business, find all matching Source-2/3 records. The metric is **per-S1 F0.5, macro-averaged over all S1**; singletons score 1 only for an empty prediction.

**Why v2.** v1 reached 0.963 on a held-out train split but ~0.94 on the leaderboard. The EDA below shows the main reason:
* test has **5.75 S2/S3 records per S1 against 4.68 in train**, while its true matches per S1 are unchanged (≈3.45);
* so about **41 % of test records match no S1, against 26 % in train**;
* a model tuned on train over-predicts on test, merging orphan records into a same-name S1 at another address.

v2 trains and tunes everything on a *test-like* train distribution, and rebuilds blocking and matching around it:

```
normalise → ghost-S1 split → sparse keys (11 families) ∪ dense bi-encoder (e5-small) → LightGBM re-rank + adaptive cut = candidate_pairs.tsv
          → 94 pair features → level-1 LightGBM (full graph, cross-fitted) + competition features
          → fine-tuned cross-encoder (multilingual-e5-base) → LightGBM stacker → global exclusive assignment + OOF threshold = matching_results.tsv
```

Every stage caches versioned artifacts, so running this notebook top to bottom resumes after the last finished stage. The source is in `er_v2/`. Execution: Lightning AI A100-40GB, 30 vCPU, 216 GB RAM.

In [1]:
import os, sys, json, subprocess, time
sys.path.insert(0, os.path.abspath('code') if os.path.isdir('code/er_v2') else os.path.abspath('.'))
import polars as pl, pandas as pd
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_width_chars(220); pl.Config.set_fmt_str_lengths(80)
from er_v2.config import OUT, FINAL_OUT, DATA, log
CE_TAGS = os.environ.get('ER_CE_TAGS', 'e5b').split(',')
print('cross-encoders:', CE_TAGS)
print('data:', DATA, '| artifacts:', OUT, '| submission:', FINAL_OUT)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

cross-encoders: ['e5b']
data: /teamspace/studios/this_studio/amlc/data | artifacts: /teamspace/studios/this_studio/amlc/output | submission: /teamspace/studios/this_studio/output
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB



## 1. Data and the train/test shift (EDA)
Group sizes, singleton rate and the per-country record/S1 ratio. The test distractor share is estimated from exact normalised-name hit rates:
* a record whose normalised name equals some S1 name is far more likely to be matched (60 %) than a distractor (31 %, India);
* solving the mixture on test India gives 3.45 matched records per S1, the same as train, so the extra test records are distractors.

In [2]:
from er_v2.normalize import read_tsv
from er_v2.config import raw_path
cnt = []
for split in ['train', 'test']:
    for s in '123':
        d = pl.scan_csv(raw_path(f'{split}_source{s}'), separator='\t', quote_char=None, infer_schema=False).group_by('country').len().collect()
        cnt += [{'split': split, 'source': f'S{s}', 'country': c, 'rows': n} for c, n in d.iter_rows()]
cnt = pl.DataFrame(cnt)
piv = cnt.pivot(on='source', index=['split', 'country'], values='rows').with_columns(((pl.col('S2') + pl.col('S3')) / pl.col('S1')).round(3).alias('records per S1'))
piv.sort('split', 'country')

split,country,S1,S2,S3,records per S1
str,str,i64,i64,i64,f64
"""test""","""France""",259452,703378,731615,5.531
"""test""","""India""",809986,2312565,2405000,5.824
"""test""","""US""",663106,1871330,1945701,5.756
"""train""","""India""",883188,2017799,2115547,4.68
"""train""","""US""",1323633,3016817,3170056,4.674


## 2. Normalisation (v2) and the ghost-S1 split

In [3]:
from er_v2 import normalize, splits
normalize.run(); splits.run()
print(json.dumps(json.load(open(OUT / 'ghost_info.json')), indent=1))
from er_v2.data import nt
nt('test', '1', ['country', 'name', 'name_core', 'name_sk', 'addr', 'addr_norm', 'loc', 'addr_nums']).sample(8, seed=1)

[   0.0 min] train_full_s1_v2.parquet cached


[   0.0 min] train_s2_v2.parquet cached


[   0.0 min] train_s3_v2.parquet cached


[   0.0 min] test_s1_v2.parquet cached


[   0.0 min] test_s2_v2.parquet cached


[   0.0 min] test_s3_v2.parquet cached


[   0.0 min] splits cached


{
 "train S1 full": 2206821,
 "ghost S1": 441365,
 "train S1 kept": 1765456,
 "records": 10320219,
 "records per kept S1": 5.846,
 "matched share (kept)": 0.5921,
 "distractor share (kept)": 0.4079,
 "distractor share (full)": 0.2599
}


country,name,name_core,name_sk,addr,addr_norm,loc,addr_nums
str,str,str,str,str,str,list[str],list[str]
"""US""","""24 Hour Massage Group""","""24 hour massage group""","""grp hr ms""","""433 Carson Springs Road, Newport, TN""","""433 carson springs rd newport tn""","[""newport""]","[""433""]"
"""US""","""Heartland Ampercap LLC""","""heartland ampercap""","""anprkp hrtlnd""","""16905 E Park Drive, Cleveland, OH""","""16905 e park dr cleveland oh""","[""cleveland""]","[""16905""]"
"""France""","""Publique Etablissements SAS""","""publique etablissements""","""etblsmnts pblk""","""120 Boulevard de l’Atlantique, La Teste-de-Buch, Nouvelle-Aquitaine""","""120 blvd de l atlantique la teste de buch nouvelle aquitaine""","[""teste"", ""buch"", … ""aquitaine""]","[""120""]"
"""India""","""Nsm Wares LLP""","""nsm wares""","""nsm vrs""","""B406The Great Eastern Chs, Plot 66 Sec 15 Cbdbelapur, Thane, Maharashtra""","""b406 the great eastern chs plot 66 sec 15 cbdbelapur thane mh""","[""thane""]","[""406"", ""66"", ""15""]"
"""France""","""Maison Fabrique SAS""","""maison fabrique""","""fbrk msn""","""Nouvelle-Aquitaine, 143 Rue Émile Combes, Mérignac""","""nouvelle aquitaine 143 rue emile combes merignac""","[""nouvelle"", ""aquitaine"", ""merignac""]","[""143""]"
"""India""","""High Media Private Limited""","""high media""","""hg md""","""New Delhi, D-226 First Floor Lal Kuan, M.B Road, Delhi""","""new delhi 226 first fl lal kuan m b rd dl""","[""new"", ""delhi""]","[""226""]"
"""US""","""Pediatric Dentistry Mountain Medicine""","""pediatric dentistry mountain medicine""","""dntstr mdsn mntn pdtrk""","""3619 Greenwood Drive, Johnson City, TN""","""3619 greenwood dr johnson city tn""","[""johnson"", ""city""]","[""3619""]"
"""US""","""Livingstone Corporation""","""livingstone""","""lngstn""","""TX, 2622 Commonwealth Street, Houston""","""tx 2622 commonwealth st houston""","[""houston""]","[""2622""]"


## 3. Candidate generation: sparse keys ∪ dense retrieval → learned cut

In [4]:
from er_v2 import blocking_sparse, dense, stage2
blocking_sparse.run(); dense.run(); stage2.run()
print('rr_tune (stage-2 tuning):'); display(pl.read_csv(OUT / 'blocking_report_rr_tune.csv'))
print('val (never used by blocking):'); display(pl.read_csv(OUT / 'blocking_report_val.csv'))
cfg = json.load(open(OUT / 'models' / 'stage2_cfg_s2a.json')); print({k: v for k, v in cfg.items() if k != 'RR_FEATS'})

[   0.1 min] train/India: cached


[   0.1 min] train/US: cached


[   0.1 min] test/France: cached


[   0.1 min] test/India: cached


[   0.1 min] test/US: cached


...

rr_tune (stage-2 tuning):


stage,S1,candidates,cand / S1,pair recall,pair precision,oracle F0.5
str,i64,i64,f64,f64,f64,f64
"""union (input to stage 2)""",100001,8674761,86.747,0.9986,0.0399,0.99959
"""sparse-only top-1""",100001,100000,1.0,0.2524,0.8738,0.64101
"""sparse-only top-3""",100001,299995,3.0,0.6423,0.7412,0.89943
"""sparse-only top-5""",100001,499974,5.0,0.8405,0.582,0.95425
"""sparse-only top-10""",100001,999738,9.997,0.9257,0.3205,0.97353
"""sparse-only top-20""",100001,1997913,19.979,0.9456,0.1639,0.98005
"""sparse-only top-50""",100001,4975930,49.759,0.9574,0.0666,0.9841
"""dense-only top-5""",100001,500005,5.0,0.9042,0.626,0.98344
"""dense-only top-10""",100001,1000010,10.0,0.9902,0.3428,0.99741


val (never used by blocking):


stage,S1,candidates,cand / S1,pair recall,pair precision,oracle F0.5
str,i64,i64,f64,f64,f64,f64
"""sparse top-50""",100000,4977097,49.771,0.9575,0.0666,0.98418
"""dense top-40""",100000,4000000,40.0,0.9977,0.0864,0.99932
"""union""",100000,8671090,86.711,0.9986,0.0399,0.99959
"""stage 2 (final candidates)""",100000,535320,5.353,0.9918,0.6414,0.99792
"""stage 2 [India]""",40020,218567,5.461,0.9918,0.6294,0.99799
"""stage 2 [US]""",59980,316753,5.281,0.9918,0.6498,0.99787


{'kmax': 8, 'floor': 0.01, 'TOL': 0.002, 'tune_oracle': 0.99796, 'tune_cand_per_s1': 5.343, 'union_oracle': 0.99959, 'union_cand_per_s1': 86.747, 'best_iteration': 681}


## 4. Pair features, level-1 model and competition features

In [5]:
from er_v2 import features, level1
features.run(); level1.run()
print(json.load(open(OUT / 'models' / 'level1_f2' / 'info.json')))

[   0.2 min] train_f2.parquet cached


[   0.2 min] test_f2.parquet cached


[   0.2 min] level-1 cached


{'iters': [277, 188, 224], 'oof_auc': 0.996783960675138}


## 5. Cross-encoder, stacker, decision rule, submission
* The cross-encoders are fine-tuned on the `ce_train` candidates.
* They score the evaluation set E = stack_train ∪ val ∪ closure (every S1 competing for their records) and all of test.
* The LightGBM stacker is fitted with 5-fold OOF on stack_train. The threshold under global exclusive assignment is chosen on OOF.
* val is used only for the report.

In [6]:
from er_v2 import stack
stack.CE_TAGS = CE_TAGS
stack.run()

[   0.2 min] evaluation set: 300,000 S1 (stack_train+val) + closure -> 851,739 S1, 5,152,337 pairs


[   0.2 min] CE training pairs 2,137,708 (pos rate 0.642)


[   0.2 min] CE e5b cached


[   0.2 min] CE stage done (0.0 min)


[   0.3 min] stacker iters [300, 290, 277, 272, 281], OOF AUC 0.99895


...

[   0.6 min] stack_train OOF: threshold+exclusive best t=0.69 F=0.98996 | expected-F+exclusive F=0.98990 -> threshold+exclusive


[   0.6 min] VAL (ghost, test-like) macro F0.5: {'all': 0.9901302839495328, 'US': 0.9894321996545956, 'India': 0.9911765382226536, 'singletons': 0.9940913160250672} | oracle 0.99792 | cand/S1 5.353


[   0.8 min] VAL ablations: CE-only {'e5b': 0.9853832230136852} | level-1 (features only) 0.98200


[   1.3 min] TEST by country:
shape: (3, 5)
┌─────────┬────────┬─────────┬─────────────┬─────────┐
│ country ┆ S1     ┆ pred/S1 ┆ empty share ┆ cand/S1 │
│ ---     ┆ ---    ┆ ---     ┆ ---         ┆ ---     │
│ str     ┆ u32    ┆ f64     ┆ f64         ┆ f64     │
╞═════════╪════════╪═════════╪═════════════╪═════════╡
│ France  ┆ 259452 ┆ 3.34    ┆ 0.0563      ┆ 6.008   │
│ India   ┆ 809986 ┆ 3.38    ┆ 0.0577      ┆ 5.932   │
│ US      ┆ 663106 ┆ 3.412   ┆ 0.0582      ┆ 5.8     │
└─────────┴────────┴─────────┴─────────────┴─────────┘


[   1.3 min] submission written: /teamspace/studios/this_studio/output (history /teamspace/studios/this_studio/amlc/output/submission_history/20260926_221126)


## 6. Decision-rule study (chosen on stack_train OOF; val for reporting only)

In [7]:
from er_v2 import decide
decide.TAG = '_'.join(CE_TAGS)
study = decide.run(write=False)
pd.DataFrame({k: {'oof': v['oof'], 'val': v['val']['all'] if isinstance(v['val'], dict) else v['val']} for k, v in study.items()}).T

[   2.0 min] A_global {"t": 0.69, "oof": 0.9899564813434281, "val": {"all": 0.9901302839495328, "US": 0.9894321996545943, "India": 0.9911765382226522, "singletons": 0.9940913160250672}}


[   2.0 min] B_per_country {"t": {"India": 0.69, "US": 0.73}, "oof": 0.9899703568434842, "val": {"all": 0.9900523814685197, "US": 0.9893023188926521, "India": 0.9911765382226516, "singletons": 0.9942703670546106}}


[   2.0 min] C_margin {"t": 0.69, "margin": 0.3, "oof": 0.9899863538916072, "val": {"all": 0.9901744681449475, "India": 0.9912393019795013, "US": 0.9894639871503027, "singletons": 0.9940913160250672}}


[   2.0 min] D_expected_f {"oof": 0.9899048859366545, "val": {"all": 0.990284678596086, "India": 0.9913273590244597, "US": 0.989588978850444, "singletons": 0.9903312444046554}}


[   2.0 min] best rule on stack_train OOF: C_margin


,oof,val
A_global,0.989956,0.990130
B_per_country,0.989970,0.990052
C_margin,0.989986,0.990174
D_expected_f,0.989905,0.990285


## 7. Results summary

In [8]:
hist = sorted((OUT / 'submission_history').iterdir())[-1]
info = json.load(open(hist / 'run_info.json'))
rows = [
    ('v1 (submitted, LB ~0.94)', 'e5-small CE + LightGBM', 'raw train (26 % distractors)', 0.96310, 0.97149, 4.79),
    ('v2: level-1 LightGBM, features only', 'LightGBM', 'ghost train (41 % distractors)', info['level1_only_val'], info['val_oracle']['oracle F0.5'], info['val_oracle']['cand / S1']),
    ('v2: cross-encoder only', 'multilingual-e5-base CE', 'ghost train', list(info['ce_only_val'].values())[0], info['val_oracle']['oracle F0.5'], info['val_oracle']['cand / S1']),
    ('v2: FINAL stack', 'LightGBM stack + CE(s)', 'ghost train', info['val']['all'], info['val_oracle']['oracle F0.5'], info['val_oracle']['cand / S1']),
]
display(pd.DataFrame(rows, columns=['experiment', 'model', 'validation distribution', 'val macro F0.5', 'oracle F0.5', 'cand / S1']))
print('val by country:', {k: round(v, 5) for k, v in info['val'].items()})
display(pd.DataFrame(info['test_by_country']))

,experiment,model,validation distribution,val macro F0.5,oracle F0.5,cand / S1
0,"v1 (submitted, LB ~0.94)",e5-small CE + LightGBM,raw train (26 % distractors),0.963100,0.97149,4.790
1,"v2: level-1 LightGBM, features only",LightGBM,ghost train (41 % distractors),0.982001,0.99792,5.353
2,v2: cross-encoder only,multilingual-e5-base CE,ghost train,0.985383,0.99792,5.353
3,v2: FINAL stack,LightGBM stack + CE(s),ghost train,0.990130,0.99792,5.353


val by country: {'all': 0.99013, 'US': 0.98943, 'India': 0.99118, 'singletons': 0.99409}


,country,S1,pred/S1,empty share,cand/S1
0,France,259452,3.340,0.0563,6.008
1,India,809986,3.380,0.0577,5.932
2,US,663106,3.412,0.0582,5.800


## 8. Official validator

In [9]:
r = subprocess.run([sys.executable, 'code/validate_submission.py', '--matching', str(FINAL_OUT / 'matching_results.tsv'),
                    '--candidate', str(FINAL_OUT / 'candidate_pairs.tsv'), '--test-dir', str(DATA), '--check-ids'], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])

ML Challenge 2026 — submission validator
  test dir: /teamspace/studios/this_studio/amlc/data
  required S1 entities: 1732544
  valid S2/S3 match IDs: 9969589
  matching_results.tsv: 1732544 rows (99969 empty, 1632575 non-empty).
  candidate_pairs.tsv: 1732544 rows (4987 empty, 1727557 non-empty).

PASS — no blocking issues found. Safe to submit.
 
